# LeetCode 981 — Time Based Key-Value Store bằng Python

Link đề: https://leetcode.com/problems/time-based-key-value-store/ · Độ khó: **Medium** · Chủ đề: Hash Table (bảng băm), String (chuỗi), Binary Search (tìm kiếm nhị phân), Design (thiết kế)

Skeleton LeetCode:

```python
class TimeMap:

    def __init__(self):
        ...

    def set(self, key: str, value: str, timestamp: int) -> None:
        ...

    def get(self, key: str, timestamp: int) -> str:
        ...


# Your TimeMap object will be instantiated and called as such:
# obj = TimeMap()
# obj.set(key,value,timestamp)
# param_2 = obj.get(key,timestamp)
```

Ghi chú: đây là bài **thiết kế lớp**, không phải một hàm: LeetCode tạo `TimeMap` rồi gọi một **chuỗi lệnh** `set` / `get`, kết quả của lệnh `set` là `null`. Notebook này **tự dựng bộ chạy chuỗi lệnh** (`run_ops`) giống cách LeetCode làm. Mỗi khóa có **lịch sử các cặp (thời điểm, giá trị)** với thời điểm của `set` **tăng ngặt**, nên `get(key, timestamp)` là bài **tìm kiếm nhị phân**: tìm cặp có thời điểm lớn nhất mà vẫn `<= timestamp`.

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: lưu **nhiều** giá trị cho cùng một khóa theo thời gian và truy vấn "giá trị hiệu lực tại thời điểm `t`".
2. Nắm các công cụ Python cần thiết: `dict` / `collections.defaultdict(list)`, `bisect.bisect_right`, tìm kiếm nhị phân tự viết, lát cắt, và mô hình **lớp có trạng thái** với chuỗi lệnh.
3. Hiểu nguyên lý: vì thời điểm của `set` **tăng ngặt**, danh sách thời điểm của mỗi khóa **luôn được sắp xếp**; `get` là "tìm phần tử cuối cùng `<= t`" bằng `bisect_right(...) - 1`.
4. Đi qua 4 cách: quét tuyến tính toàn bộ nhật ký (brute force), hai danh sách song song kèm `bisect_right` (cách nộp chính), tìm kiếm nhị phân tự viết, và danh sách các cặp kèm tham số `key=` của `bisect`.
5. Tránh các lỗi kinh điển: `bisect_left` thay vì `bisect_right`, quên trừ `1`, **chỉ số âm** khi `t` nhỏ hơn mọi thời điểm (Python trả về phần tử cuối!), gộp mọi khóa vào một danh sách chung, và chỉ tra đúng thời điểm `t` (bỏ qua "nhỏ hơn hoặc bằng").

## 2. Đề bài & ví dụ

Thiết kế một cấu trúc dữ liệu key-value theo thời gian: lưu **nhiều giá trị** cho cùng một khóa ở **các thời điểm khác nhau** và truy xuất giá trị của khóa tại một thời điểm.

Cài đặt lớp `TimeMap`:

- `TimeMap()` khởi tạo đối tượng.
- `void set(String key, String value, int timestamp)`: lưu khóa `key` với giá trị `value` tại thời điểm `timestamp`.
- `String get(String key, int timestamp)`: trả về một giá trị mà `set` đã được gọi trước đó với `timestamp_prev <= timestamp`. Nếu có nhiều giá trị như vậy, trả về giá trị ứng với `timestamp_prev` **lớn nhất**. Nếu không có giá trị nào, trả về `""`.

**Ví dụ 1:**

```
["TimeMap", "set", "get", "get", "set", "get", "get"]
[[], ["foo", "bar", 1], ["foo", 1], ["foo", 3], ["foo", "bar2", 4], ["foo", 4], ["foo", 5]]
```

Output: `[null, null, "bar", "bar", null, "bar2", "bar2"]`

Giải thích: `set("foo","bar",1)`; `get("foo",1)` là `"bar"`; `get("foo",3)` là `"bar"` (không có giá trị tại 3 và 2, giá trị gần nhất tại thời điểm 1); `set("foo","bar2",4)`; `get("foo",4)` là `"bar2"`; `get("foo",5)` là `"bar2"`.

**Ràng buộc:**

- `1 <= key.length, value.length <= 100`
- `key` và `value` gồm chữ thường và chữ số.
- `1 <= timestamp <= 10^7`
- Mọi `timestamp` của `set` **tăng ngặt**.
- Tối đa `2 * 10^5` lần gọi `set` và `get`.

In [1]:
from typing import List, Optional, Tuple, Dict
from collections import defaultdict
from bisect import bisect_left, bisect_right, insort
import random, time

def run_ops(cls, ops: List[str], args: list, class_name: str = "TimeMap") -> list:
    """Mô phỏng cách LeetCode gọi lớp: phần tử đầu là tên lớp (constructor), các phần tử sau là tên method.
    Trả về list kết quả, None ứng với null."""
    obj = None
    out = []
    for i, (op, arg) in enumerate(zip(ops, args)):
        if i == 0:
            assert op == class_name, (op, class_name)
            obj = cls(*arg)
            out.append(None)
        else:
            out.append(getattr(obj, op)(*arg))
    return out

example_ops = ["TimeMap", "set", "get", "get", "set", "get", "get"]
example_args = [[], ["foo", "bar", 1], ["foo", 1], ["foo", 3], ["foo", "bar2", 4], ["foo", 4], ["foo", 5]]
example_expected = [None, None, "bar", "bar", None, "bar2", "bar2"]
print("ops     =", example_ops)
print("kỳ vọng =", example_expected)

ops     = ['TimeMap', 'set', 'get', 'get', 'set', 'get', 'get']
kỳ vọng = [None, None, 'bar', 'bar', None, 'bar2', 'bar2']


## 3. Kiến thức nền

### 3.1. `dict` và `defaultdict(list)` cho dữ liệu theo khóa

`d[key]` là danh sách lịch sử của `key`. Với `defaultdict(list)`, truy cập khóa chưa có tự tạo danh sách rỗng, nhưng cũng **tạo khóa mới mỗi lần đọc**; ở hàm `get` ta nên dùng `d.get(key)` (hoặc kiểm tra `key in d`) để không vô tình thêm khóa.

In [2]:
d = defaultdict(list)
d["foo"].append((1, "bar"))
print(dict(d), "| 'zzz' in d:", "zzz" in d)
_ = d["zzz"]                      # chỉ đọc thôi mà đã tạo khóa mới
print("sau khi đọc d['zzz']:", dict(d))
print("d.get('qqq') không tạo khóa:", d.get("qqq"), "| 'qqq' in d:", "qqq" in d)

{'foo': [(1, 'bar')]} | 'zzz' in d: False
sau khi đọc d['zzz']: {'foo': [(1, 'bar')], 'zzz': []}
d.get('qqq') không tạo khóa: None | 'qqq' in d: False


### 3.2. `bisect_right` và `bisect_left`

Với danh sách `a` đã sắp xếp tăng:

- `bisect_right(a, x)`: chỉ số **sau** mọi phần tử `<= x` (vị trí chèn bên phải của `x`);
- `bisect_left(a, x)`: chỉ số **trước** mọi phần tử `>= x`.

Số phần tử `<= x` là `bisect_right(a, x)`, nên **phần tử cuối cùng `<= x`** nằm ở chỉ số `bisect_right(a, x) - 1` (và `-1` nghĩa là **không có**).

In [3]:
a = [1, 4, 7]
for x in (0, 1, 3, 4, 5, 7, 9):
    r, l = bisect_right(a, x), bisect_left(a, x)
    print(f"x={x}: bisect_right={r}, bisect_left={l} -> phần tử cuối <= x ở chỉ số {r - 1}" + (" (không có)" if r == 0 else f" (giá trị {a[r - 1]})"))

x=0: bisect_right=0, bisect_left=0 -> phần tử cuối <= x ở chỉ số -1 (không có)
x=1: bisect_right=1, bisect_left=0 -> phần tử cuối <= x ở chỉ số 0 (giá trị 1)
x=3: bisect_right=1, bisect_left=1 -> phần tử cuối <= x ở chỉ số 0 (giá trị 1)
x=4: bisect_right=2, bisect_left=1 -> phần tử cuối <= x ở chỉ số 1 (giá trị 4)
x=5: bisect_right=2, bisect_left=2 -> phần tử cuối <= x ở chỉ số 1 (giá trị 4)
x=7: bisect_right=3, bisect_left=2 -> phần tử cuối <= x ở chỉ số 2 (giá trị 7)
x=9: bisect_right=3, bisect_left=3 -> phần tử cuối <= x ở chỉ số 2 (giá trị 7)


### 3.3. Cạm bẫy chỉ số âm của Python

`a[-1]` là **phần tử cuối**, không phải "ngoài mảng". Nếu `bisect_right(...) - 1` ra `-1` (không có thời điểm nào `<= t`) mà ta vẫn truy cập `a[-1]` thì nhận về phần tử **cuối cùng**: sai và **không báo lỗi**. Luôn kiểm tra `idx < 0` trước khi truy cập.

In [4]:
vals = ["bar", "bar2"]
idx = bisect_right([1, 4], 0) - 1
print("idx =", idx, "| vals[idx] =", vals[idx], "(sai, đáng lẽ phải là rỗng!)")

idx = -1 | vals[idx] = bar2 (sai, đáng lẽ phải là rỗng!)


## 4. Ý tưởng nền tảng / trực giác (tính tay)

Ví dụ 1. Lịch sử của khóa `foo` sau các lệnh `set`: `[(1, "bar"), (4, "bar2")]`, tức thời điểm `[1, 4]` và giá trị `["bar", "bar2"]`.

- `get("foo", 1)`: các thời điểm `<= 1` là `[1]`, lấy cái lớn nhất là `1` -> `"bar"`.
- `get("foo", 3)`: các thời điểm `<= 3` là `[1]` -> `"bar"`.
- `get("foo", 4)`: các thời điểm `<= 4` là `[1, 4]`, lớn nhất là `4` -> `"bar2"`.
- `get("foo", 5)`: `[1, 4]` -> `"bar2"`.
- `get("foo", 0)` (thử thêm): không có thời điểm nào `<= 0` -> `""`.

Vì các thời điểm của `set` **tăng dần**, danh sách thời điểm của mỗi khóa luôn **đã sắp xếp**, nên ta có thể tìm "thời điểm lớn nhất `<= t`" bằng **tìm kiếm nhị phân**.

## 5. Cách 1 — Brute Force: nhật ký phẳng và quét toàn bộ

**Ý tưởng.** Giữ **một danh sách phẳng** `log` chứa mọi lệnh `set` dưới dạng `(key, value, timestamp)`. `get(key, t)` quét **toàn bộ** `log`, giữ lại bản ghi cùng `key` có `timestamp <= t` và `timestamp` lớn nhất. Không dùng tính chất sắp xếp nào cả, nên đây cũng là **oracle độc lập** để kiểm các cách sau.

In [5]:
class TimeMapLog:
    def __init__(self):
        self.log: List[Tuple[str, str, int]] = []

    def set(self, key: str, value: str, timestamp: int) -> None:
        self.log.append((key, value, timestamp))

    def get(self, key: str, timestamp: int) -> str:
        best = None
        for k, v, t in self.log:
            if k == key and t <= timestamp and (best is None or t > best[0]):
                best = (t, v)
        return "" if best is None else best[1]

got = run_ops(TimeMapLog, example_ops, example_args)
print(got, "OK" if got == example_expected else "SAI")
assert got == example_expected

[None, None, 'bar', 'bar', None, 'bar2', 'bar2'] OK


**Độ phức tạp và nút thắt.** `set` là `O(1)`, nhưng mỗi `get` quét **mọi** lần `set` của mọi khóa: `O(S)` với `S` là số lần `set` đã gọi. Với `10^5` lần `set` và `10^5` lần `get` là `10^10` phép so sánh: quá chậm. Đo thử:

In [6]:
def gen_ops(rng: random.Random, n_ops: int, n_keys: int, set_prob: float = 0.5, max_ts: int = 10 ** 7, val_len: int = 3):
    """Sinh chuỗi lệnh hợp lệ: thời điểm của set tăng ngặt; get dùng thời điểm bất kỳ trong [1, max_ts]."""
    keys = [f"k{i}" for i in range(n_keys)]
    ops, args = ["TimeMap"], [[]]
    ts = 0
    n_sets = 0
    for i in range(n_ops):
        remaining = n_ops - i
        if rng.random() < set_prob and ts < max_ts - remaining:
            ts += rng.randint(1, max(1, min(50, (max_ts - ts) // max(remaining, 1))))
            ops.append("set")
            args.append([rng.choice(keys), "".join(rng.choice("abc123") for _ in range(rng.randint(1, val_len))), ts])
            n_sets += 1
        else:
            ops.append("get")
            args.append([rng.choice(keys + ["nokey"]), rng.randint(1, max(ts + 20, 1))])
    return ops, args

rng = random.Random(981)
for n in (500, 1000, 2000, 4000):
    ops, args = gen_ops(rng, n, 3)
    t0 = time.perf_counter()
    run_ops(TimeMapLog, ops, args)
    dt = time.perf_counter() - t0
    print(f"{n:5} lệnh: {dt * 1000:8.1f} ms  -> ngoại suy 2*10^5 lệnh (tăng bậc hai): ~{dt * (200000 / n) ** 2:9.0f} s")

  500 lệnh:      1.4 ms  -> ngoại suy 2*10^5 lệnh (tăng bậc hai): ~      232 s
 1000 lệnh:      5.0 ms  -> ngoại suy 2*10^5 lệnh (tăng bậc hai): ~      200 s
 2000 lệnh:     21.8 ms  -> ngoại suy 2*10^5 lệnh (tăng bậc hai): ~      218 s
 4000 lệnh:     84.3 ms  -> ngoại suy 2*10^5 lệnh (tăng bậc hai): ~      211 s


Nút thắt: ta quét cả những bản ghi của **khóa khác** và những bản ghi **không cần thiết**. Hai cải tiến: (1) gom bản ghi **theo khóa**; (2) trong mỗi khóa, thời điểm **đã sắp xếp** nên dùng tìm kiếm nhị phân.

## 6. Ý tưởng thuật toán / bước đột phá

**Gom theo khóa, rồi tìm kiếm nhị phân trên thời điểm.**

- `store[key] = (timestamps, values)`: hai danh sách song song; `set` chỉ cần `append` vì thời điểm tăng ngặt nên danh sách **luôn đã sắp xếp** (không cần chèn giữa).
- `get(key, t)`: `i = bisect_right(timestamps, t) - 1`; nếu `i < 0` thì `""`, ngược lại `values[i]`.

`set` là `O(1)`, `get` là `O(log m)` với `m` là số lần `set` của khóa.

| Cách | Ý tưởng |
|---|---|
| 2 | Hai danh sách song song + `bisect_right` (nộp) |
| 3 | Tự viết tìm kiếm nhị phân "phần tử cuối `<= t`" |
| 4 | Một danh sách các cặp `(timestamp, value)` + `bisect_right(..., key=...)` |

## 7. Các cách tiếp theo

### 7.1. Cách 2 — Hai danh sách song song + `bisect_right` (cách nộp chính)

**Giải thích từng đoạn.**

- `self.times = defaultdict(list)` và `self.values = defaultdict(list)`: danh sách thời điểm và giá trị của từng khóa, cùng chỉ số.
- `set`: `append` vào cả hai (thời điểm tăng ngặt nên vẫn sắp xếp).
- `get`: nếu khóa chưa có thì `""` (tránh tạo khóa mới khi đọc); `i = bisect_right(times, t) - 1`; `i < 0` thì `""`.

In [7]:
class TimeMapBisect:
    def __init__(self):
        self.times = defaultdict(list)
        self.values = defaultdict(list)

    def set(self, key: str, value: str, timestamp: int) -> None:
        self.times[key].append(timestamp)
        self.values[key].append(value)

    def get(self, key: str, timestamp: int) -> str:
        times = self.times.get(key)
        if not times:
            return ""
        i = bisect_right(times, timestamp) - 1
        return self.values[key][i] if i >= 0 else ""

got = run_ops(TimeMapBisect, example_ops, example_args)
print(got, "OK" if got == example_expected else "SAI")
assert got == example_expected

[None, None, 'bar', 'bar', None, 'bar2', 'bar2'] OK


**Code nộp LeetCode (Cách 2):**

In [8]:
class TimeMap:

    def __init__(self):
        from collections import defaultdict
        self.times = defaultdict(list)    # key -> các thời điểm (tăng ngặt)
        self.values = defaultdict(list)   # key -> các giá trị tương ứng

    def set(self, key: str, value: str, timestamp: int) -> None:
        self.times[key].append(timestamp)
        self.values[key].append(value)

    def get(self, key: str, timestamp: int) -> str:
        from bisect import bisect_right
        times = self.times.get(key)       # .get để không tạo khóa mới khi chỉ đọc
        if not times:
            return ""
        i = bisect_right(times, timestamp) - 1     # chỉ số phần tử cuối cùng <= timestamp
        return self.values[key][i] if i >= 0 else ""

TimeMapSubmit2 = TimeMap
got = run_ops(TimeMapSubmit2, example_ops, example_args)
assert got == example_expected
print("Cách 2 qua ví dụ:", got)

Cách 2 qua ví dụ: [None, None, 'bar', 'bar', None, 'bar2', 'bar2']


**Độ phức tạp (Cách 2):** `set` `O(1)` (amortized); `get` `O(log m)`; bộ nhớ `O(S)` tổng số lần `set`.

---

### 7.2. Cách 3 — Tự viết tìm kiếm nhị phân

**Ý tưởng khác.** Không dùng `bisect`; tự viết "tìm chỉ số của phần tử cuối cùng `<= t`" để nắm chắc điều kiện biên.

**Giải thích từng đoạn.** Giữ khoảng nửa mở `[lo, hi)`; bất biến: mọi chỉ số `< lo` có thời điểm `<= t`, mọi chỉ số `>= hi` có thời điểm `> t`. Mỗi vòng lấy `mid = (lo + hi) // 2`: nếu `times[mid] <= t` thì `lo = mid + 1`, ngược lại `hi = mid`. Khi kết thúc `lo == hi` là số phần tử `<= t`, nên đáp án ở chỉ số `lo - 1`.

In [9]:
def last_le(times: List[int], t: int) -> int:
    """Chỉ số của phần tử cuối cùng <= t trong danh sách tăng; -1 nếu không có."""
    lo, hi = 0, len(times)
    while lo < hi:
        mid = (lo + hi) // 2
        if times[mid] <= t:
            lo = mid + 1
        else:
            hi = mid
    return lo - 1

class TimeMapManual:
    def __init__(self):
        self.data: Dict[str, Tuple[List[int], List[str]]] = {}

    def set(self, key: str, value: str, timestamp: int) -> None:
        if key not in self.data:
            self.data[key] = ([], [])
        self.data[key][0].append(timestamp)
        self.data[key][1].append(value)

    def get(self, key: str, timestamp: int) -> str:
        if key not in self.data:
            return ""
        times, values = self.data[key]
        i = last_le(times, timestamp)
        return values[i] if i >= 0 else ""

for times in ([], [5], [1, 4, 7], [1, 4, 7, 9, 12]):
    for t in range(0, 14):
        assert last_le(times, t) == bisect_right(times, t) - 1
got = run_ops(TimeMapManual, example_ops, example_args)
print(got, "OK" if got == example_expected else "SAI")
assert got == example_expected

[None, None, 'bar', 'bar', None, 'bar2', 'bar2'] OK


**Code nộp LeetCode (Cách 3):**

In [10]:
class TimeMap:

    def __init__(self):
        self.data = {}                    # key -> (danh sách thời điểm, danh sách giá trị)

    def set(self, key: str, value: str, timestamp: int) -> None:
        if key not in self.data:
            self.data[key] = ([], [])
        self.data[key][0].append(timestamp)
        self.data[key][1].append(value)

    def get(self, key: str, timestamp: int) -> str:
        if key not in self.data:
            return ""
        times, values = self.data[key]
        lo, hi = 0, len(times)            # tìm số phần tử có thời điểm <= timestamp
        while lo < hi:
            mid = (lo + hi) // 2
            if times[mid] <= timestamp:
                lo = mid + 1
            else:
                hi = mid
        return values[lo - 1] if lo > 0 else ""

TimeMapSubmit3 = TimeMap
got = run_ops(TimeMapSubmit3, example_ops, example_args)
assert got == example_expected
print("Cách 3 qua ví dụ:", got)

Cách 3 qua ví dụ: [None, None, 'bar', 'bar', None, 'bar2', 'bar2']


**Độ phức tạp (Cách 3):** như Cách 2: `set` `O(1)`, `get` `O(log m)`, bộ nhớ `O(S)`; hằng số lớn hơn một chút vì vòng lặp viết bằng Python thay vì C.

---

### 7.3. Cách 4 — Danh sách các cặp và `bisect_right(..., key=...)`

**Ý tưởng khác.** Lưu **một danh sách các cặp** `(timestamp, value)` cho mỗi khóa (gọn hơn vì chỉ một cấu trúc) và dùng tham số `key=` của `bisect` (Python 3.10 trở lên) để tìm theo thành phần thứ nhất: `bisect_right(pairs, t, key=lambda p: p[0])`.

**Giải thích từng đoạn.**

- `key=lambda p: p[0]` cho `bisect` biết so sánh `t` với **thời điểm** của từng cặp (không phải cả cặp).
- Ví dụ khác: có thể dùng thủ thuật `bisect_right(pairs, (t, chr(127)))` để so cả cặp (giá trị gồm chữ thường và chữ số nên luôn nhỏ hơn `chr(127)`), nhưng cách dùng `key=` rõ ràng hơn.

In [11]:
class TimeMapPairs:
    def __init__(self):
        self.data = defaultdict(list)

    def set(self, key: str, value: str, timestamp: int) -> None:
        self.data[key].append((timestamp, value))

    def get(self, key: str, timestamp: int) -> str:
        pairs = self.data.get(key)
        if not pairs:
            return ""
        i = bisect_right(pairs, timestamp, key=lambda p: p[0]) - 1
        return pairs[i][1] if i >= 0 else ""

got = run_ops(TimeMapPairs, example_ops, example_args)
print(got, "OK" if got == example_expected else "SAI")
assert got == example_expected

[None, None, 'bar', 'bar', None, 'bar2', 'bar2'] OK


**Code nộp LeetCode (Cách 4):**

In [12]:
class TimeMap:

    def __init__(self):
        self.data = {}                    # key -> danh sách các cặp (timestamp, value), tăng theo timestamp

    def set(self, key: str, value: str, timestamp: int) -> None:
        self.data.setdefault(key, []).append((timestamp, value))

    def get(self, key: str, timestamp: int) -> str:
        from bisect import bisect_right
        pairs = self.data.get(key)
        if not pairs:
            return ""
        i = bisect_right(pairs, timestamp, key=lambda p: p[0]) - 1
        return pairs[i][1] if i >= 0 else ""

TimeMapSubmit4 = TimeMap
got = run_ops(TimeMapSubmit4, example_ops, example_args)
assert got == example_expected
print("Cách 4 qua ví dụ:", got)

Cách 4 qua ví dụ: [None, None, 'bar', 'bar', None, 'bar2', 'bar2']


**Độ phức tạp (Cách 4):** `set` `O(1)`, `get` `O(log m)` (mỗi so sánh gọi một hàm nhỏ nên hơi chậm hơn Cách 2), bộ nhớ `O(S)`. Cần Python `>= 3.10` cho tham số `key=` của `bisect`.

## 8. TRACE từng bước

### 8.1. Trạng thái bên trong sau mỗi lệnh (Cách 2)

In `times` và `values` của khóa `foo` sau mỗi lệnh, và chỉ số `bisect_right - 1` của mỗi `get`.

In [13]:
def trace_ops(ops, args):
    obj = TimeMapBisect()
    out = []
    for i, (op, arg) in enumerate(zip(ops, args)):
        if i == 0:
            print("khởi tạo TimeMap()")
            out.append(None)
            continue
        if op == "set":
            obj.set(*arg)
            out.append(None)
            print(f"set{tuple(arg)} -> times={dict(obj.times)}, values={dict(obj.values)}")
        else:
            key, t = arg
            times = obj.times.get(key, [])
            idx = bisect_right(times, t) - 1
            res = obj.get(key, t)
            out.append(res)
            print(f"get{tuple(arg)}: times={times}, bisect_right={idx + 1}, idx={idx} -> {res!r}")
    return out

got = trace_ops(example_ops, example_args)
assert got == example_expected

khởi tạo TimeMap()
set('foo', 'bar', 1) -> times={'foo': [1]}, values={'foo': ['bar']}
get('foo', 1): times=[1], bisect_right=1, idx=0 -> 'bar'
get('foo', 3): times=[1], bisect_right=1, idx=0 -> 'bar'
set('foo', 'bar2', 4) -> times={'foo': [1, 4]}, values={'foo': ['bar', 'bar2']}
get('foo', 4): times=[1, 4], bisect_right=2, idx=1 -> 'bar2'
get('foo', 5): times=[1, 4], bisect_right=2, idx=1 -> 'bar2'


### 8.2. Tìm kiếm nhị phân tự viết cho `times = [1, 4, 7, 9, 12]`, `t = 8`

In `lo`, `hi`, `mid` từng vòng. Kết quả đúng: phần tử cuối cùng `<= 8` là `7` ở chỉ số `2`.

In [14]:
def trace_last_le(times, t):
    lo, hi = 0, len(times)
    step = 0
    while lo < hi:
        mid = (lo + hi) // 2
        action = "lo = mid + 1" if times[mid] <= t else "hi = mid"
        print(f"vòng {step}: lo={lo}, hi={hi}, mid={mid}, times[mid]={times[mid]} {'<=' if times[mid] <= t else '>'} {t} -> {action}")
        if times[mid] <= t:
            lo = mid + 1
        else:
            hi = mid
        step += 1
    print(f"kết thúc lo=hi={lo} -> chỉ số cần tìm = lo - 1 = {lo - 1}")
    return lo - 1

assert trace_last_le([1, 4, 7, 9, 12], 8) == 2

vòng 0: lo=0, hi=5, mid=2, times[mid]=7 <= 8 -> lo = mid + 1
vòng 1: lo=3, hi=5, mid=4, times[mid]=12 > 8 -> hi = mid
vòng 2: lo=3, hi=4, mid=3, times[mid]=9 > 8 -> hi = mid
kết thúc lo=hi=3 -> chỉ số cần tìm = lo - 1 = 2


### 8.3. Trường hợp `t` nhỏ hơn mọi thời điểm

Với `times = [5, 9]` và `t = 3`: `bisect_right = 0`, chỉ số `-1` nghĩa là **không có** giá trị; kết quả phải là `""`. Nếu lỡ viết `values[-1]` sẽ nhận phần tử cuối (xem lỗi số 3 ở mục 13).

In [15]:
tm = TimeMapBisect()
tm.set("a", "x", 5)
tm.set("a", "y", 9)
print("get('a', 3) =", repr(tm.get("a", 3)), "| get('a', 5) =", repr(tm.get("a", 5)), "| get('a', 8) =", repr(tm.get("a", 8)), "| get('a', 100) =", repr(tm.get("a", 100)))
assert (tm.get("a", 3), tm.get("a", 5), tm.get("a", 8), tm.get("a", 100)) == ("", "x", "x", "y")

get('a', 3) = '' | get('a', 5) = 'x' | get('a', 8) = 'x' | get('a', 100) = 'y'


## 9. Code nộp LeetCode

Ta chọn **Cách 2** (hai danh sách song song + `bisect_right`): `set` `O(1)`, `get` `O(log m)`.

```python
class TimeMap:

    def __init__(self):
        from collections import defaultdict
        self.times = defaultdict(list)    # key -> các thời điểm (tăng ngặt)
        self.values = defaultdict(list)   # key -> các giá trị tương ứng

    def set(self, key: str, value: str, timestamp: int) -> None:
        self.times[key].append(timestamp)
        self.values[key].append(value)

    def get(self, key: str, timestamp: int) -> str:
        from bisect import bisect_right
        times = self.times.get(key)       # .get để không tạo khóa mới khi chỉ đọc
        if not times:
            return ""
        i = bisect_right(times, timestamp) - 1     # chỉ số phần tử cuối cùng <= timestamp
        return self.values[key][i] if i >= 0 else ""
```

In [16]:
class TimeMap:

    def __init__(self):
        from collections import defaultdict
        self.times = defaultdict(list)    # key -> các thời điểm (tăng ngặt)
        self.values = defaultdict(list)   # key -> các giá trị tương ứng

    def set(self, key: str, value: str, timestamp: int) -> None:
        self.times[key].append(timestamp)
        self.values[key].append(value)

    def get(self, key: str, timestamp: int) -> str:
        from bisect import bisect_right
        times = self.times.get(key)       # .get để không tạo khóa mới khi chỉ đọc
        if not times:
            return ""
        i = bisect_right(times, timestamp) - 1     # chỉ số phần tử cuối cùng <= timestamp
        return self.values[key][i] if i >= 0 else ""

got = run_ops(TimeMap, example_ops, example_args)
print(got, "OK" if got == example_expected else "SAI")
assert got == example_expected

[None, None, 'bar', 'bar', None, 'bar2', 'bar2'] OK


## 10. Vì sao đúng

**Bất biến cấu trúc.** Với mỗi khóa, `times` và `values` **cùng độ dài**, `values[j]` là giá trị của lần `set` có thời điểm `times[j]`, và `times` **tăng ngặt** (vì thời điểm của mọi `set` toàn cục tăng ngặt, nên cũng tăng ngặt khi chỉ xét một khóa). Điều này đúng ngay từ lúc rỗng và được `append` giữ nguyên.

**Tính đúng của `get`.** Tập các lần `set` thỏa `timestamp_prev <= t` của khóa là một **tiền tố** `times[0..k-1]` của danh sách tăng, trong đó `k = bisect_right(times, t)` là số phần tử `<= t`. Nếu `k = 0` thì không có giá trị: trả `""`. Ngược lại `timestamp_prev` **lớn nhất** là phần tử cuối của tiền tố, tức `times[k - 1]`, và giá trị của nó là `values[k - 1]`. Đó chính xác là yêu cầu của đề. Khóa chưa từng `set` thì không có giá trị: trả `""`.

**Độ phức tạp.** `set` `O(1)` amortized; `get` `O(log m)` nhờ tìm kiếm nhị phân; tổng `O(S + G log S)` cho `S` lần `set` và `G` lần `get`.

**Cách 3** viết tay đúng nhờ bất biến vòng lặp (mọi chỉ số `< lo` có `times <= t`, mọi chỉ số `>= hi` có `times > t`); khi `lo == hi` thì `lo` là số phần tử `<= t`.

**Kiểm chứng bằng máy.** Oracle độc lập (Cách 1) là **nhật ký phẳng** quét toàn bộ, không dùng sắp xếp hay tìm kiếm nhị phân. Ta chạy **cùng một chuỗi lệnh** qua mọi lớp bằng `run_ops` rồi so sánh danh sách kết quả; chuỗi lệnh sinh ngẫu nhiên có seed, đúng ràng buộc (thời điểm `set` tăng ngặt), kể cả `get` ở thời điểm trước mọi `set`, đúng bằng một thời điểm đã `set`, và sau tất cả.

In [17]:
def make_classes():
    return {"log": TimeMapLog, "bisect": TimeMapBisect, "manual": TimeMapManual, "pairs": TimeMapPairs,
            "submit2": TimeMapSubmit2, "submit3": TimeMapSubmit3, "submit4": TimeMapSubmit4, "final": TimeMap}

def compare_all(ops, args, expected=None):
    ref = run_ops(TimeMapLog, ops, args)
    for name, cls in make_classes().items():
        got = run_ops(cls, ops, args)
        assert got == ref, (name, ops[:12], args[:12])
    if expected is not None:
        assert ref == expected, (ref, expected)
    return ref

rng = random.Random(1981)
n_get = n_hit = 0
for _ in range(400):
    ops, args = gen_ops(rng, rng.randint(1, 60), rng.randint(1, 4))
    ref = compare_all(ops, args)
    for op, r in zip(ops, ref):
        if op == "get":
            n_get += 1
            n_hit += r != ""
print(f"400 chuỗi lệnh ngẫu nhiên: mọi lớp khớp oracle; {n_get} lệnh get, trong đó {n_hit} có giá trị, {n_get - n_hit} trả rỗng")
assert 0 < n_hit < n_get

400 chuỗi lệnh ngẫu nhiên: mọi lớp khớp oracle; 6305 lệnh get, trong đó 2645 có giá trị, 3660 trả rỗng


## 11. Time / Space Complexity

| Cách | `set` | `get` | Space |
|---|---|---|---|
| 1. Nhật ký phẳng | `O(1)` | `O(S)` | `O(S)` |
| 2. Hai danh sách + `bisect_right` | `O(1)` | `O(log m)` | `O(S)` |
| 3. Tìm kiếm nhị phân tự viết | `O(1)` | `O(log m)` | `O(S)` |
| 4. Danh sách cặp + `key=` | `O(1)` | `O(log m)` | `O(S)` |

Đo ở cỡ tối đa: `10^5` lần `set` và `10^5` lần `get` (tổng `2 * 10^5` lệnh); thời điểm `set` tăng ngặt, `get` ngẫu nhiên.

In [18]:
def big_case(rng, n_sets=100000, n_gets=100000, n_keys=5):
    keys = [f"key{i}" for i in range(n_keys)]
    ts = 0
    ops, args = ["TimeMap"], [[]]
    for i in range(n_sets):
        ts += rng.randint(1, 90)                       # <= 9 * 10^6 < 10^7
        ops.append("set")
        args.append([rng.choice(keys), "v" + str(i), ts])
    for _ in range(n_gets):
        ops.append("get")
        args.append([rng.choice(keys), rng.randint(1, ts + 100)])
    return ops, args

rng = random.Random(11)
ops, args = big_case(rng)
ref = None
for name in ("bisect", "manual", "pairs", "final"):
    cls = make_classes()[name]
    t0 = time.perf_counter()
    got = run_ops(cls, ops, args)
    dt = time.perf_counter() - t0
    if ref is None:
        ref = got
    assert got == ref, name
    print(f"{name:7}: {len(ops) - 1} lệnh trong {dt * 1000:8.1f} ms")
print("số get có giá trị:", sum(1 for r, o in zip(ref, ops) if o == "get" and r != ""), "/ 100000")

bisect : 200000 lệnh trong    111.6 ms


manual : 200000 lệnh trong    240.4 ms
pairs  : 200000 lệnh trong    192.4 ms


final  : 200000 lệnh trong    126.3 ms
số get có giá trị: 99992 / 100000


## 12. Bộ test

Gồm: ví dụ của đề, edge case có kỳ vọng viết tay (kèm lý do), phản ví dụ cho các cách sai (tìm bằng tìm kiếm so với oracle), đối chiếu mọi lớp với oracle nhật ký phẳng trên chuỗi lệnh ngẫu nhiên có seed, test cỡ lớn, và các tính chất.

In [19]:
# 12.1 ví dụ đề
compare_all(example_ops, example_args, example_expected)
print("12.1 ví dụ đề: OK")

# 12.2 edge case (kỳ vọng viết tay, kèm lý do)
edge = [
    # get trước mọi set: rỗng
    (["TimeMap", "get"], [[], ["a", 5]], [None, ""]),
    # khóa không tồn tại
    (["TimeMap", "set", "get"], [[], ["a", "x", 3], ["b", 3]], [None, None, ""]),
    # get đúng thời điểm set, trước, sau
    (["TimeMap", "set", "get", "get", "get"], [[], ["a", "x", 3], ["a", 2], ["a", 3], ["a", 4]], [None, None, "", "x", "x"]),
    # hai giá trị: chọn thời điểm lớn nhất <= t
    (["TimeMap", "set", "set", "get", "get", "get", "get"],
     [[], ["a", "x", 2], ["a", "y", 6], ["a", 1], ["a", 2], ["a", 5], ["a", 6]], [None, None, None, "", "x", "x", "y"]),
    # hai khóa xen kẽ không lẫn nhau
    (["TimeMap", "set", "set", "set", "get", "get", "get"],
     [[], ["a", "A1", 1], ["b", "B2", 2], ["a", "A3", 3], ["a", 2], ["b", 3], ["b", 1]], [None, None, None, None, "A1", "B2", ""]),
    # cùng giá trị ở hai thời điểm
    (["TimeMap", "set", "set", "get", "get"], [[], ["a", "x", 1], ["a", "x", 5], ["a", 3], ["a", 9]], [None, None, None, "x", "x"]),
    # biên thời điểm: 1 và 10^7
    (["TimeMap", "set", "set", "get", "get", "get"],
     [[], ["a", "lo", 1], ["a", "hi", 10 ** 7], ["a", 10 ** 7 - 1], ["a", 10 ** 7], ["a", 1]], [None, None, None, "lo", "hi", "lo"]),
    # chỉ có đúng một set; get ở nhiều thời điểm
    (["TimeMap", "set", "get", "get", "get"], [[], ["z", "v", 10], ["z", 1], ["z", 10], ["z", 11]], [None, None, "", "v", "v"]),
    # value dài 100 ký tự, key dài 100 ký tự
    (["TimeMap", "set", "get"], [[], ["k" * 100, "v" * 100, 7], ["k" * 100, 7]], [None, None, "v" * 100]),
]
for ops, args, expected in edge:
    compare_all(ops, args, expected)
print("12.2 edge case: OK", len(edge), "ca")

12.1 ví dụ đề: OK
12.2 edge case: OK 9 ca


In [20]:
# 12.3 phản ví dụ: các cách sai (tìm bằng tìm kiếm có seed so với oracle nhật ký phẳng)
class WrongBisectLeft:                          # bisect_left thay vì bisect_right
    def __init__(self):
        self.times, self.values = defaultdict(list), defaultdict(list)
    def set(self, key, value, timestamp):
        self.times[key].append(timestamp); self.values[key].append(value)
    def get(self, key, timestamp):
        times = self.times.get(key)
        if not times:
            return ""
        i = bisect_left(times, timestamp) - 1
        return self.values[key][i] if i >= 0 else ""

class WrongNoMinusOne:                          # quên trừ 1: trả giá trị SAU thời điểm
    def __init__(self):
        self.times, self.values = defaultdict(list), defaultdict(list)
    def set(self, key, value, timestamp):
        self.times[key].append(timestamp); self.values[key].append(value)
    def get(self, key, timestamp):
        times = self.times.get(key)
        if not times:
            return ""
        i = bisect_right(times, timestamp)
        return self.values[key][i] if i < len(times) else ""

class WrongNegativeIndex:                       # không kiểm tra i < 0: values[-1] là phần tử cuối
    def __init__(self):
        self.times, self.values = defaultdict(list), defaultdict(list)
    def set(self, key, value, timestamp):
        self.times[key].append(timestamp); self.values[key].append(value)
    def get(self, key, timestamp):
        times = self.times.get(key)
        if not times:
            return ""
        i = bisect_right(times, timestamp) - 1
        return self.values[key][i]

class WrongOneList:                             # gộp mọi khóa vào một danh sách chung
    def __init__(self):
        self.times, self.values = [], []
    def set(self, key, value, timestamp):
        self.times.append(timestamp); self.values.append(value)
    def get(self, key, timestamp):
        i = bisect_right(self.times, timestamp) - 1
        return self.values[i] if i >= 0 else ""

class WrongExactOnly:                           # chỉ tra đúng thời điểm (bỏ qua "nhỏ hơn hoặc bằng")
    def __init__(self):
        self.d = {}
    def set(self, key, value, timestamp):
        self.d[(key, timestamp)] = value
    def get(self, key, timestamp):
        return self.d.get((key, timestamp), "")

def find_counter(cls, seed=9, tries=20000):
    rng = random.Random(seed)
    for _ in range(tries):
        ops, args = gen_ops(rng, rng.randint(2, 10), rng.randint(1, 2), val_len=2, max_ts=30)
        try:
            got = run_ops(cls, ops, args)
        except Exception:
            return ops, args
        if got != run_ops(TimeMapLog, ops, args):
            return ops, args
    return None

counters = {}
for name, cls in (("bisect_left thay vì bisect_right", WrongBisectLeft), ("quên trừ 1", WrongNoMinusOne),
                  ("chỉ số âm (không kiểm tra i < 0)", WrongNegativeIndex), ("gộp mọi khóa một danh sách", WrongOneList),
                  ("chỉ tra đúng thời điểm", WrongExactOnly)):
    case = find_counter(cls)
    assert case is not None, name
    counters[name] = case
    ops, args = case
    print(f"{name:34} ops={ops[1:]} args={args[1:]}\n{'':34} sai -> {run_ops(cls, ops, args)[1:]} | đúng -> {run_ops(TimeMapLog, ops, args)[1:]}")
    compare_all(ops, args)
print("12.3 phản ví dụ: OK")

bisect_left thay vì bisect_right   ops=['set', 'get', 'get', 'set', 'set', 'get', 'set', 'get', 'get', 'get'] args=[['k0', 'a', 3], ['nokey', 15], ['k0', 5], ['k0', 'b3', 6], ['k0', '23', 10], ['nokey', 2], ['k0', 'c', 15], ['k0', 3], ['nokey', 16], ['nokey', 7]]
                                   sai -> [None, '', 'a', None, None, '', None, '', '', ''] | đúng -> [None, '', 'a', None, None, '', None, 'a', '', '']
quên trừ 1                         ops=['set', 'get', 'get', 'get', 'get', 'set', 'get', 'get', 'get'] args=[['k0', '21', 1], ['k0', 11], ['nokey', 2], ['k0', 15], ['k1', 6], ['k0', '2', 2], ['nokey', 3], ['k1', 4], ['k0', 22]]
                                   sai -> [None, '', '', '', '', None, '', '', ''] | đúng -> [None, '21', '', '21', '', None, '', '', '2']
chỉ số âm (không kiểm tra i < 0)   ops=['get', 'set', 'set', 'get', 'get', 'set'] args=[['k0', 1], ['k0', 'c2', 4], ['k0', '1', 7], ['k0', 3], ['nokey', 23], ['k0', '3', 21]]
                                   sai ->

In [21]:
# 12.4 ngẫu nhiên có seed: nhiều kích thước, số khóa, độ dài giá trị, xác suất set
rng = random.Random(4981)
for _ in range(600):
    ops, args = gen_ops(rng, rng.randint(1, 200), rng.randint(1, 6), set_prob=rng.choice([0.2, 0.5, 0.8]),
                        max_ts=rng.choice([300, 10 ** 4, 10 ** 7]), val_len=rng.randint(1, 5))
    compare_all(ops, args)
print("12.4 ngẫu nhiên (600 chuỗi lệnh): OK")

# 12.5 cỡ lớn: 10^5 set + get lẫn lộn: mọi lớp nhanh khớp nhau; oracle chỉ kiểm trên một mẫu get
rng = random.Random(5981)
keys = [f"key{i}" for i in range(4)]
ts = 0
ops, args = ["TimeMap"], [[]]
for i in range(60000):
    if rng.random() < 0.5:
        ts += rng.randint(1, 100)
        ops.append("set")
        args.append([rng.choice(keys), "v" + str(i), ts])
    else:
        ops.append("get")
        args.append([rng.choice(keys + ["none"]), rng.randint(1, ts + 100)])
fast = {n: c for n, c in make_classes().items() if n != "log"}
outs = {n: run_ops(c, ops, args) for n, c in fast.items()}
assert len({tuple(o) for o in outs.values()}) == 1
sample = TimeMapLog()
mism = 0
for op, arg, r in zip(ops[1:], args[1:], outs["final"][1:]):
    if op == "set":
        sample.set(*arg)
    elif rng.random() < 0.01:                    # kiểm oracle trên ~1% lệnh get (oracle chậm)
        mism += sample.get(*arg) != r
assert mism == 0
print("12.5 cỡ lớn (60000 lệnh): OK")

# 12.6 tính chất: giá trị trả về luôn thuộc tập giá trị đã set; get đơn điệu theo thời điểm; get tại thời điểm set trả đúng giá trị đó
rng = random.Random(6981)
for _ in range(200):
    tm = TimeMap()
    history = defaultdict(list)
    ts = 0
    for _ in range(rng.randint(1, 60)):
        ts += rng.randint(1, 10)
        key = rng.choice(["a", "b", "c"])
        tm.set(key, f"v{ts}", ts)                 # giá trị duy nhất theo thời điểm: nhận ra thời điểm từ giá trị
        history[key].append(ts)
    for key, tss in history.items():
        prev_ts = 0
        for t in range(0, ts + 12):
            r = tm.get(key, t)
            if r == "":
                assert t < tss[0]
            else:
                got_ts = int(r[1:])
                assert got_ts in tss and got_ts <= t and got_ts >= prev_ts       # đơn điệu không giảm
                assert not any(got_ts < x <= t for x in tss)                      # là thời điểm lớn nhất <= t
                prev_ts = got_ts
        for x in tss:
            assert tm.get(key, x) == f"v{x}"
    assert tm.get("zzz", 5) == "" and "zzz" not in tm.times                       # đọc không tạo khóa mới
print("12.6 tính chất: OK")

12.4 ngẫu nhiên (600 chuỗi lệnh): OK


12.5 cỡ lớn (60000 lệnh): OK
12.6 tính chất: OK


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Dùng `bisect_left` thay vì `bisect_right`

`bisect_left(times, t)` cho vị trí **trước** các phần tử bằng `t`; trừ 1 sẽ bỏ qua phần tử có đúng thời điểm `t`.

In [22]:
ops, args = counters["bisect_left thay vì bisect_right"]
print("ops =", ops[1:], "args =", args[1:])
print("sai ->", run_ops(WrongBisectLeft, ops, args)[1:], "| đúng ->", run_ops(TimeMap, ops, args)[1:])
assert run_ops(WrongBisectLeft, ops, args) != run_ops(TimeMap, ops, args)
print("ví dụ đề: get('foo', 1) sai ->", run_ops(WrongBisectLeft, example_ops, example_args)[2], "(đúng: 'bar')")

ops = ['set', 'get', 'get', 'set', 'set', 'get', 'set', 'get', 'get', 'get'] args = [['k0', 'a', 3], ['nokey', 15], ['k0', 5], ['k0', 'b3', 6], ['k0', '23', 10], ['nokey', 2], ['k0', 'c', 15], ['k0', 3], ['nokey', 16], ['nokey', 7]]
sai -> [None, '', 'a', None, None, '', None, '', '', ''] | đúng -> [None, '', 'a', None, None, '', None, 'a', '', '']
ví dụ đề: get('foo', 1) sai ->  (đúng: 'bar')


Vì sao sai: đề cho phép `timestamp_prev <= timestamp` (**bằng** cũng được). **Đúng:** `bisect_right`.

### Lỗi 2 — Quên trừ 1

`bisect_right` trả về **số** phần tử `<= t`, không phải chỉ số của phần tử cuối cùng `<= t`. Dùng trực tiếp sẽ lấy phần tử **sau**.

In [23]:
ops, args = counters["quên trừ 1"]
print("ops =", ops[1:], "args =", args[1:])
print("sai ->", run_ops(WrongNoMinusOne, ops, args)[1:], "| đúng ->", run_ops(TimeMap, ops, args)[1:])
assert run_ops(WrongNoMinusOne, ops, args) != run_ops(TimeMap, ops, args)

ops = ['set', 'get', 'get', 'get', 'get', 'set', 'get', 'get', 'get'] args = [['k0', '21', 1], ['k0', 11], ['nokey', 2], ['k0', 15], ['k1', 6], ['k0', '2', 2], ['nokey', 3], ['k1', 4], ['k0', 22]]
sai -> [None, '', '', '', '', None, '', '', ''] | đúng -> [None, '21', '', '21', '', None, '', '', '2']


Vì sao sai: chỉ số `k = bisect_right(...)` trỏ tới phần tử đầu tiên **lớn hơn** `t`. **Đúng:** `i = bisect_right(...) - 1`.

### Lỗi 3 — Quên kiểm tra chỉ số âm

Khi `t` nhỏ hơn mọi thời điểm, `bisect_right - 1 = -1` và `values[-1]` là **phần tử cuối**: sai mà không báo lỗi.

In [24]:
ops, args = counters["chỉ số âm (không kiểm tra i < 0)"]
print("ops =", ops[1:], "args =", args[1:])
print("sai ->", run_ops(WrongNegativeIndex, ops, args)[1:], "| đúng ->", run_ops(TimeMap, ops, args)[1:])
assert run_ops(WrongNegativeIndex, ops, args) != run_ops(TimeMap, ops, args)

ops = ['get', 'set', 'set', 'get', 'get', 'set'] args = [['k0', 1], ['k0', 'c2', 4], ['k0', '1', 7], ['k0', 3], ['nokey', 23], ['k0', '3', 21]]
sai -> ['', None, None, '1', '', None] | đúng -> ['', None, None, '', '', None]


Vì sao sai: `-1` hợp lệ như một chỉ số Python nhưng nghĩa là "cuối cùng". **Đúng:** `return values[i] if i >= 0 else ""`.

### Lỗi 4 — Gộp mọi khóa vào một danh sách chung

Nếu bỏ qua `key`, giá trị trả về có thể thuộc **khóa khác**.

In [25]:
ops, args = counters["gộp mọi khóa một danh sách"]
print("ops =", ops[1:], "args =", args[1:])
print("sai ->", run_ops(WrongOneList, ops, args)[1:], "| đúng ->", run_ops(TimeMap, ops, args)[1:])
assert run_ops(WrongOneList, ops, args) != run_ops(TimeMap, ops, args)

ops = ['set', 'get', 'get', 'get', 'get', 'set', 'get', 'get', 'get'] args = [['k0', '21', 1], ['k0', 11], ['nokey', 2], ['k0', 15], ['k1', 6], ['k0', '2', 2], ['nokey', 3], ['k1', 4], ['k0', 22]]
sai -> [None, '21', '21', '21', '21', None, '2', '2', '2'] | đúng -> [None, '21', '', '21', '', None, '', '', '2']


Vì sao sai: mỗi khóa có lịch sử riêng. **Đúng:** `dict` từ khóa sang danh sách.

### Lỗi 5 — Chỉ tra đúng thời điểm

Dùng `dict[(key, timestamp)]` rồi `get` đúng thời điểm: bỏ qua ý "giá trị còn hiệu lực cho tới khi bị ghi đè". Ví dụ 1 có `get("foo", 3)` trả `"bar"` dù chưa từng `set` tại thời điểm `3`.

In [26]:
ops, args = counters["chỉ tra đúng thời điểm"]
print("ops =", ops[1:], "args =", args[1:])
print("sai ->", run_ops(WrongExactOnly, ops, args)[1:], "| đúng ->", run_ops(TimeMap, ops, args)[1:])
assert run_ops(WrongExactOnly, ops, args) != run_ops(TimeMap, ops, args)
print("ví dụ đề: sai ->", run_ops(WrongExactOnly, example_ops, example_args), "(đúng:", example_expected, ")")

ops = ['set', 'get', 'get', 'get', 'get', 'set', 'get', 'get', 'get'] args = [['k0', '21', 1], ['k0', 11], ['nokey', 2], ['k0', 15], ['k1', 6], ['k0', '2', 2], ['nokey', 3], ['k1', 4], ['k0', 22]]
sai -> [None, '', '', '', '', None, '', '', ''] | đúng -> [None, '21', '', '21', '', None, '', '', '2']
ví dụ đề: sai -> [None, None, 'bar', '', None, 'bar2', ''] (đúng: [None, None, 'bar', 'bar', None, 'bar2', 'bar2'] )


Vì sao sai: `get` phải tìm thời điểm **lớn nhất `<= t`**, không phải đúng bằng `t`. **Đúng:** tìm kiếm nhị phân trên danh sách thời điểm đã sắp xếp.

## 14. Mẹo & nhận dạng pattern

- **Dấu hiệu:** "giá trị hiệu lực tại thời điểm `t`, thời điểm ghi tăng dần" ⇒ danh sách thời điểm **đã sắp xếp** + **tìm kiếm nhị phân** ("phần tử cuối cùng `<= t`").
- Công thức nhớ: **phần tử cuối `<= t`** là `bisect_right(a, t) - 1`; **phần tử đầu `>= t`** là `bisect_left(a, t)`.
- Với `defaultdict`, dùng `.get(key)` khi **chỉ đọc** để không tạo khóa rỗng.
- Cẩn thận chỉ số `-1` trong Python (phần tử cuối), luôn kiểm tra `i >= 0`.
- Bài thiết kế lớp: tự dựng bộ chạy chuỗi lệnh (`run_ops`) và so sánh với một cài đặt **ngây thơ nhưng hiển nhiên đúng** (oracle).
- Cùng họ: Snapshot Array (1146), Online Election (911), My Calendar (729, 731), Design Hit Counter (362).
- Test nên có: `get` trước mọi `set`, `get` đúng bằng thời điểm, `get` sau tất cả, khóa lạ, nhiều khóa xen kẽ, thời điểm `1` và `10^7`.

## 15. Biến thể

1. **`set` với thời điểm bất kỳ (không tăng):** dùng `bisect.insort` để chèn đúng chỗ vào danh sách đã sắp xếp (`O(m)` mỗi lần chèn), `get` vẫn `O(log m)`.
2. **`get` trả về giá trị có thời điểm nhỏ nhất `>= t`** (kế tiếp thay vì trước đó): dùng `bisect_left`.
3. **Xóa giá trị / hết hạn sau `ttl` đơn vị thời gian.**
4. Bài liên quan: 1146, 911, 729.

Biến thể 1 (thời điểm bất kỳ, mỗi khóa không trùng thời điểm) kiểm bằng oracle quét toàn bộ:

In [27]:
class TimeMapAnyOrder:
    def __init__(self):
        self.data = defaultdict(list)          # key -> danh sách (timestamp, value) luôn sắp xếp

    def set(self, key: str, value: str, timestamp: int) -> None:
        insort(self.data[key], (timestamp, value))

    def get(self, key: str, timestamp: int) -> str:
        pairs = self.data.get(key)
        if not pairs:
            return ""
        i = bisect_right(pairs, timestamp, key=lambda p: p[0]) - 1
        return pairs[i][1] if i >= 0 else ""

rng = random.Random(15)
for _ in range(300):
    a, b = TimeMapAnyOrder(), TimeMapLog()
    used = set()
    for _ in range(rng.randint(1, 40)):
        if rng.random() < 0.5:
            key, t = rng.choice("ab"), rng.randint(1, 60)
            if (key, t) in used:
                continue
            used.add((key, t))
            v = f"v{t}{key}"
            a.set(key, v, t); b.set(key, v, t)
        else:
            key, t = rng.choice("abz"), rng.randint(0, 70)
            assert a.get(key, t) == b.get(key, t)
print("biến thể thời điểm bất kỳ (insort): khớp oracle")

biến thể thời điểm bất kỳ (insort): khớp oracle


## 16. So sánh các cách

| Cách | Ý tưởng | `set` | `get` | Space | Khi nào dùng |
|---|---|---|---|---|---|
| 1. Nhật ký phẳng | Quét mọi bản ghi | `O(1)` | `O(S)` | `O(S)` | Oracle |
| 2. Hai danh sách + `bisect_right` | Nhị phân trên thời điểm | `O(1)` | `O(log m)` | `O(S)` | **Cách nộp chính** |
| 3. Nhị phân tự viết | Bất biến `[lo, hi)` | `O(1)` | `O(log m)` | `O(S)` | Học điều kiện biên |
| 4. Danh sách cặp + `key=` | Một cấu trúc gọn | `O(1)` | `O(log m)` | `O(S)` | Python `>= 3.10` |

## 17. Bài tập vận dụng

Làm tuần tự từ dễ đến khó. Các cell có chữ `TODO` **không được gọi** khi chạy notebook; bạn tự điền rồi bỏ dấu `#` ở các dòng `assert` để kiểm tra. Đáp án nằm ở mục 19.

### Bài 1 — Đoán output (dễ)

Không chạy code, hãy đoán rồi chạy để kiểm tra:

```python
a = [2, 5, 5, 9]
print(bisect_right(a, 5), bisect_left(a, 5), bisect_right(a, 1), bisect_right(a, 100))
print(last_le(a, 5), last_le(a, 1), last_le(a, 8))
```

In [28]:
# Bài 1: ghi dự đoán của bạn
my_guess_1 = None      # bốn số của dòng 1
my_guess_2 = None      # ba số của dòng 2

### Bài 2 — Tính tay (dễ-vừa)

Chuỗi lệnh sau (chỉ ghi các lệnh sau constructor): `set("a","x",2)`, `set("b","y",3)`, `set("a","z",8)`, `get("a",2)`, `get("a",7)`, `get("a",9)`, `get("b",2)`, `get("c",5)`. Ghi list kết quả của **năm lệnh `get`** theo thứ tự.

In [29]:
# Bài 2: điền kết quả tính tay (list 5 chuỗi)
hand_answer = None

### Bài 3 — Điền chỗ trống (vừa)

Hoàn thành `TimeMap` với hai danh sách song song và `bisect_right`.

In [30]:
class ExerciseTimeMap3:
    def __init__(self):
        self.times = defaultdict(list)
        self.values = defaultdict(list)

    def set(self, key: str, value: str, timestamp: int) -> None:
        # TODO 1: thêm timestamp vào times[key] và value vào values[key]
        pass

    def get(self, key: str, timestamp: int) -> str:
        # TODO 2: nếu khóa chưa có thì trả ""
        # TODO 3: i = chỉ số phần tử cuối cùng <= timestamp (bisect_right rồi trừ 1)
        # TODO 4: nếu i < 0 trả "", ngược lại trả values[key][i]
        pass


# assert run_ops(ExerciseTimeMap3, example_ops, example_args) == example_expected

### Bài 4 — Tự viết lại (vừa-khó)

Không nhìn lại Cách 3, tự viết `exercise_4_last_le(times, t)` (tìm kiếm nhị phân, **không** dùng `bisect`) trả về chỉ số của phần tử cuối cùng `<= t`, hoặc `-1` nếu không có.

In [31]:
def exercise_4_last_le(times: List[int], t: int) -> int:
    # TODO
    pass


# for times in ([], [5], [1, 4, 7], [1, 4, 7, 9, 12]):
#     for t in range(0, 14):
#         assert exercise_4_last_le(times, t) == bisect_right(times, t) - 1

### Bài 5 — Mini challenge: đếm số lần `set` trong khoảng thời gian (khó)

Viết lớp `ExerciseTimeMap5` có `set(key, value, timestamp)` như đề và thêm `count(key, t1, t2)` trả về **số lần `set` của `key` có `t1 <= timestamp <= t2`** (`0` nếu khóa chưa có; `t1 <= t2`). Mỗi `count` phải `O(log m)`.

```python
tm = ExerciseTimeMap5()
tm.set("a", "x", 2); tm.set("a", "y", 5); tm.set("a", "z", 9); tm.set("b", "w", 4)
tm.count("a", 2, 9)  -> 3
tm.count("a", 3, 8)  -> 1
tm.count("a", 6, 8)  -> 0
tm.count("b", 1, 100) -> 1
tm.count("c", 1, 100) -> 0
```
Gợi ý: `bisect_right(times, t2) - bisect_left(times, t1)`.

In [32]:
class ExerciseTimeMap5:
    def __init__(self):
        self.times = defaultdict(list)

    def set(self, key: str, value: str, timestamp: int) -> None:
        # TODO
        pass

    def count(self, key: str, t1: int, t2: int) -> int:
        # TODO
        pass


# tm = ExerciseTimeMap5()
# tm.set("a", "x", 2); tm.set("a", "y", 5); tm.set("a", "z", 9); tm.set("b", "w", 4)
# assert [tm.count("a", 2, 9), tm.count("a", 3, 8), tm.count("a", 6, 8), tm.count("b", 1, 100), tm.count("c", 1, 100)] == [3, 1, 0, 1, 0]

## 18. Gợi ý 3 mức

**Bài 1**
- Mức 1: `bisect_right` trả về số phần tử `<= x`; `bisect_left` trả về số phần tử `< x`.
- Mức 2: với `x = 5` trong `[2, 5, 5, 9]`: `bisect_right = 3`, `bisect_left = 1`; `last_le` bằng `bisect_right - 1`.
- Mức 3: dòng 1: `3 1 0 4`; dòng 2: `2 -1 2`.

**Bài 2**
- Mức 1: khóa `a` có thời điểm `[2, 8]` và giá trị `["x", "z"]`; khóa `b` có `[3]`, `["y"]`.
- Mức 2: `get("a",2)` -> `x`; `get("a",7)` -> `x` (8 > 7); `get("a",9)` -> `z`; `get("b",2)` -> rỗng (3 > 2); `get("c",5)` -> rỗng (khóa lạ).
- Mức 3: `["x", "x", "z", "", ""]`.

**Bài 3**
- Mức 1: `self.times[key].append(timestamp)` và `self.values[key].append(value)`.
- Mức 2: `times = self.times.get(key)`; `if not times: return ""`.
- Mức 3: `i = bisect_right(times, timestamp) - 1`; `return self.values[key][i] if i >= 0 else ""`.

**Bài 4**
- Mức 1: giữ khoảng `[lo, hi)`, ban đầu `lo = 0`, `hi = len(times)`.
- Mức 2: `mid = (lo + hi) // 2`; nếu `times[mid] <= t` thì `lo = mid + 1` ngược lại `hi = mid`.
- Mức 3: trả về `lo - 1`.

**Bài 5**
- Mức 1: `set` chỉ cần `self.times[key].append(timestamp)`.
- Mức 2: dùng `.get(key)` để khóa lạ trả `0`.
- Mức 3: `bisect_right(times, t2) - bisect_left(times, t1)`.

## 19. Đáp án tham khảo

Bạn nên tự làm trước khi xem phần này. Mỗi cell dưới đây đều tự kiểm tra bằng `assert`.

### Đáp án Bài 1

In [33]:
a = [2, 5, 5, 9]
print(bisect_right(a, 5), bisect_left(a, 5), bisect_right(a, 1), bisect_right(a, 100))
print(last_le(a, 5), last_le(a, 1), last_le(a, 8))

assert (bisect_right(a, 5), bisect_left(a, 5), bisect_right(a, 1), bisect_right(a, 100)) == (3, 1, 0, 4)
assert (last_le(a, 5), last_le(a, 1), last_le(a, 8)) == (2, -1, 2)

3 1 0 4
2 -1 2


### Đáp án Bài 2

In [34]:
ops = ["TimeMap", "set", "set", "set", "get", "get", "get", "get", "get"]
args = [[], ["a", "x", 2], ["b", "y", 3], ["a", "z", 8], ["a", 2], ["a", 7], ["a", 9], ["b", 2], ["c", 5]]
res = run_ops(TimeMap, ops, args)
hand_answer_solution = ["x", "x", "z", "", ""]
assert res[4:] == hand_answer_solution == run_ops(TimeMapLog, ops, args)[4:]
print("Bài 2 OK:", res[4:])

Bài 2 OK: ['x', 'x', 'z', '', '']


### Đáp án Bài 3

In [35]:
class ExerciseTimeMap3Solution:
    def __init__(self):
        self.times = defaultdict(list)
        self.values = defaultdict(list)

    def set(self, key: str, value: str, timestamp: int) -> None:
        self.times[key].append(timestamp)
        self.values[key].append(value)

    def get(self, key: str, timestamp: int) -> str:
        times = self.times.get(key)
        if not times:
            return ""
        i = bisect_right(times, timestamp) - 1
        return self.values[key][i] if i >= 0 else ""

assert run_ops(ExerciseTimeMap3Solution, example_ops, example_args) == example_expected
rng = random.Random(3)
for _ in range(100):
    ops, args = gen_ops(rng, rng.randint(1, 60), 3)
    assert run_ops(ExerciseTimeMap3Solution, ops, args) == run_ops(TimeMapLog, ops, args)
print("Bài 3 OK")

Bài 3 OK


### Đáp án Bài 4

In [36]:
def exercise_4_last_le_solution(times: List[int], t: int) -> int:
    lo, hi = 0, len(times)
    while lo < hi:
        mid = (lo + hi) // 2
        if times[mid] <= t:
            lo = mid + 1
        else:
            hi = mid
    return lo - 1

for times in ([], [5], [1, 4, 7], [1, 4, 7, 9, 12]):
    for t in range(0, 14):
        assert exercise_4_last_le_solution(times, t) == bisect_right(times, t) - 1
rng = random.Random(4)
for _ in range(300):
    times = sorted(rng.sample(range(1, 100), rng.randint(0, 20)))
    t = rng.randint(0, 105)
    assert exercise_4_last_le_solution(times, t) == bisect_right(times, t) - 1
print("Bài 4 OK")

Bài 4 OK


### Đáp án Bài 5

In [37]:
class ExerciseTimeMap5Solution:
    def __init__(self):
        self.times = defaultdict(list)

    def set(self, key: str, value: str, timestamp: int) -> None:
        self.times[key].append(timestamp)

    def count(self, key: str, t1: int, t2: int) -> int:
        times = self.times.get(key)
        if not times:
            return 0
        return bisect_right(times, t2) - bisect_left(times, t1)

tm = ExerciseTimeMap5Solution()
tm.set("a", "x", 2); tm.set("a", "y", 5); tm.set("a", "z", 9); tm.set("b", "w", 4)
assert [tm.count("a", 2, 9), tm.count("a", 3, 8), tm.count("a", 6, 8), tm.count("b", 1, 100), tm.count("c", 1, 100)] == [3, 1, 0, 1, 0]
rng = random.Random(5)
for _ in range(200):
    tm = ExerciseTimeMap5Solution()
    log = []
    ts = 0
    for _ in range(rng.randint(0, 30)):
        ts += rng.randint(1, 5)
        key = rng.choice("ab")
        tm.set(key, "v", ts)
        log.append((key, ts))
    key, t1 = rng.choice("abc"), rng.randint(0, ts + 3)
    t2 = rng.randint(t1, ts + 5)
    assert tm.count(key, t1, t2) == sum(1 for k, t in log if k == key and t1 <= t <= t2)
print("Bài 5 OK")

Bài 5 OK


## 20. Template nhớ nhanh + Checklist trước khi submit

### Template (lịch sử theo khóa + tìm kiếm nhị phân)

```python
class TimeMap:

    def __init__(self):
        from collections import defaultdict
        self.times = defaultdict(list)    # key -> thời điểm tăng ngặt
        self.values = defaultdict(list)   # key -> giá trị tương ứng

    def set(self, key: str, value: str, timestamp: int) -> None:
        self.times[key].append(timestamp)
        self.values[key].append(value)

    def get(self, key: str, timestamp: int) -> str:
        from bisect import bisect_right
        times = self.times.get(key)       # .get: chỉ đọc, không tạo khóa mới
        if not times:
            return ""
        i = bisect_right(times, timestamp) - 1     # phần tử cuối cùng <= timestamp
        return self.values[key][i] if i >= 0 else ""
```

Công thức nhớ nhanh: phần tử cuối `<= t` ở chỉ số `bisect_right(a, t) - 1`; chỉ số `-1` nghĩa là không có.

### Checklist trước khi submit

- [ ] Dùng `bisect_right` (không `bisect_left`) cho điều kiện `<= timestamp`?
- [ ] Đã trừ `1` sau `bisect_right`, và kiểm tra `i >= 0` trước khi truy cập (tránh `values[-1]`)?
- [ ] Khóa lạ trả `""` và dùng `.get(key)` để không tạo khóa mới khi đọc?
- [ ] Mỗi khóa có danh sách riêng (không gộp chung)?
- [ ] `set` chỉ `append` (vì thời điểm tăng ngặt): danh sách vẫn sắp xếp?
- [ ] Đã thử: `get` trước mọi `set`, đúng bằng thời điểm `set`, sau tất cả, khóa lạ, nhiều khóa xen kẽ, thời điểm `1` và `10^7`?